# Week 3, day 2 — Worksheet 10 SOLUTIONS: encoding and capstone   (L05)

Executed in the lab image (pandas 3.0.5) against the real files in `data/`.
Every quoted number is what it actually printed.

Question 2 is the one to re-read. The deck shows this producing zeros and ones,
and it does not.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 10 — Encoding, sampling, capstone. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")
cust = pd.read_csv("data/customers_messy.csv")

print("orders:", orders.shape, "| customers:", cust.shape)
print("categories:", list(orders["Category"].unique()))

PART A — one-hot encoding

### Question 1

Three columns — `Category_Furniture`, `Category_Office Supplies`, `Category_Technology` — shape `(1093, 3)`.

One column per category, one row per original row, exactly one `True` in
each row. The column names are `<original>_<value>`, so a space in the
category becomes a space in the column name — `Category_Office Supplies`
needs bracket access, not `df.Category_Office Supplies`.

In [ ]:
dm = pd.get_dummies(orders[["Category"]], columns=["Category"])
print(dm.head().to_string())
print()
print("columns:", list(dm.columns))
print("shape:", dm.shape)

### Question 2

dtypes are **`bool`**, printing as `True`/`False`. -> `dtype=int` gives the deck's `0`/`1`.

The deck's slide shows the cells as `0` and `1`, and it was right when it
was written. **Pandas 2.0 changed `get_dummies` to return booleans**, and
this is 3.0.5.

The values are equivalent and the dtype is not. Booleans use an eighth of
the memory, which is why the change was made, and they behave differently in
places you might not expect: concatenated with a numeric frame they may
upcast, written to CSV they become `True`/`False` text rather than `1`/`0`,
and some older scikit-learn code checks for numeric dtypes.

Pass `dtype=int` when the consumer expects numbers. Just know you are
asking for it.

In [ ]:
dm = pd.get_dummies(orders[["Category"]], columns=["Category"])
print("dtypes:", set(dm.dtypes.astype(str)))
print()
print(dm.head(3).to_string())
print()
as_int = pd.get_dummies(orders[["Category"]], columns=["Category"], dtype=int)
print("with dtype=int:")
print(as_int.head(3).to_string())
print("dtypes:", set(as_int.dtypes.astype(str)))

### Question 3

Column sums: `Office Supplies 606`, `Technology 324`, `Furniture 163`. -> identical for the bool and int versions, and matching `value_counts()`.

Summing booleans counts the `True`s, so the dtype makes no difference to
the arithmetic — `True` is 1 in any numeric context.

The sums matching `value_counts()` is the check that the encoding is
complete: every row got exactly one `True`, so nothing was dropped and
nothing was double-counted.

In [ ]:
dm = pd.get_dummies(orders[["Category"]], columns=["Category"])
as_int = pd.get_dummies(orders[["Category"]], columns=["Category"], dtype=int)
print("bool sums:")
print(dm.sum().to_string())
print()
print("int sums identical:", dm.sum().equals(as_int.sum()))
print()
print("value_counts:")
print(orders["Category"].value_counts().to_string())

### Question 4

`drop_first=True` -> 2 columns instead of 3, `Furniture` removed. -> **163** rows are all-`False`, matching the **163** Furniture rows exactly.

No information was lost. With three mutually exclusive categories, two
columns determine the third: all-`False` means Furniture, and the count
proves it.

That redundancy is why the option exists — perfectly collinear columns
break the matrix inversion in linear models. For tree models and for
human-readable output it is unnecessary and makes the table harder to read,
because one category has no column and you have to know which.

Use it when the consumer is a linear model. Otherwise leave it off.

In [ ]:
full = pd.get_dummies(orders[["Category"]], columns=["Category"])
dropped = pd.get_dummies(orders[["Category"]], columns=["Category"], drop_first=True)
print("full:   ", list(full.columns))
print("dropped:", list(dropped.columns))
print()
print("shapes:", full.shape, "vs", dropped.shape)
print()
# The dropped category is the one where every remaining column is False.
implied = (~dropped.any(axis=1)).sum()
print("rows with all-False (the dropped category):", implied)
print("Furniture rows in the source:", (orders["Category"] == "Furniture").sum())

PART B — sampling

### Question 5

`random_state=42` twice -> the same five OrderIDs, `[31781, 58725, 32000, 41634, 3845]`. -> seed `7` gives different rows.

`random_state` makes the sample reproducible, which is what turns a random
choice into something a colleague can rerun and a reviewer can check.

Without it the sample changes every run, so a bug that only appears on
certain rows is unreproducible and a 'the numbers moved' report is
uninvestigable.

In [ ]:
a = orders.sample(n=5, random_state=42)["OrderID"].tolist()
b = orders.sample(n=5, random_state=42)["OrderID"].tolist()
c = orders.sample(n=5, random_state=7)["OrderID"].tolist()
print("seed 42:", a)
print("seed 42:", b, "| identical:", a == b)
print("seed 7: ", c, "| identical to 42:", a == c)

### Question 6

`sample(frac=1)` reorders the rows -> the sorted OrderIDs and the total `Sales` are unchanged.

A full-fraction sample is a shuffle: every row exactly once, in a new
order. Useful before splitting a file into train and test, because a file
ordered by date, region or ID gives you a split that is not random at all —
and this one *is* ordered by region.

The two checks are worth keeping as a habit. Same rows, same total: the
shuffle moved things without losing or duplicating any.

In [ ]:
shuffled = orders.sample(frac=1, random_state=42).reset_index(drop=True)
print("before:", orders["OrderID"].head(3).tolist())
print("after: ", shuffled["OrderID"].head(3).tolist())
print()
print("same rows:", sorted(orders["OrderID"]) == sorted(shuffled["OrderID"]))
print("same total:", round(orders["Sales"].sum(), 2) == round(shuffled["Sales"].sum(), 2))

### Question 7

A 10% sample of 109 rows. -> `West` is `21.2%` of the file and **`14.7%`** of the sample; `Ontario` `27.6%` -> `32.1%`; `Nunavut` `0.8%` -> `1.8%`.

A 6.5-point swing on West and more than double on Nunavut, from a
perfectly correct random sample.

Small categories are where this bites. Nunavut has 9 orders in the whole
file; a 10% sample expects one, and getting two doubles its apparent share.
Any statistic you compute per region from this sample is built on single
digits.

If the proportions have to hold — and for a train/test split on an
imbalanced target they usually do — a plain random sample will not do it.
Sample within group (`groupby(...).sample(frac=0.1)`), or use a stratified
split.

In [ ]:
s = orders.sample(frac=0.1, random_state=42)
full_pct = (orders["Region"].value_counts(normalize=True) * 100).round(1)
samp_pct = (s["Region"].value_counts(normalize=True) * 100).round(1)
comp = pd.DataFrame({"full": full_pct, "sample": samp_pct}).fillna(0)
comp["diff"] = (comp["sample"] - comp["full"]).round(1)
print("sample rows:", len(s))
print(comp.to_string())

PART C — capstone: the whole day, in order

### Question 8

`440` rows with `17` missing -> after replace, **`50`** missing -> after strip and dedupe, **`400`** rows and `400` distinct customers.

The whole cleaning sequence, with a count printed at every step so nothing
moves without you seeing it.

The order is deliberate and it is the order worksheets 06 and 07 argued
for: **normalise the values first, then remove the duplicates.** Strip
before dedupe or the 11 near-duplicates survive; replace the markers before
you count missing or the number is wrong by 33.

Ending on 400 rows and 400 distinct IDs is the check that the file is now
one row per customer — the grain you can actually join on.

In [ ]:
import numpy as np
df = pd.read_csv("data/customers_messy.csv")
print("loaded:            %d rows, %d missing Province"
      % (len(df), df["Province"].isna().sum()))

df["Province"] = df["Province"].replace(["-", "?"], np.nan)
print("after replace:     %d rows, %d missing Province"
      % (len(df), df["Province"].isna().sum()))

df["CustomerName"] = df["CustomerName"].str.strip()
print("after strip:       %d rows" % len(df))

df = df.drop_duplicates()
print("after dedupe:      %d rows" % len(df))
print()
print("distinct customers:", df["CustomerID"].nunique())

### Question 9

**0** unbinned rows, and the `Region` x `Band` table's grand total is `1605576.22`, matching the raw total exactly.

The two checks that make a summary table trustworthy, both cheap.

**Nothing fell outside the bins** — the `float("inf")` top edge from
worksheet 08 Q5, without which 19 orders and 274,216 of revenue would have
vanished from this table with no error.

**The grand total reconciles with the source.** That only works because the
aggregate is a sum; it is the one check that catches a fan-out, a dropped
group, or a silently discarded band.

`observed=True` is there because `Band` is a categorical — without it the
table would carry rows for band-region combinations that have no orders.

In [ ]:
work = orders.copy()
work["Band"] = pd.cut(work["Sales"], bins=[0, 100, 500, 2000, float("inf")],
                      labels=["small", "medium", "large", "huge"])
print("unbinned rows:", work["Band"].isna().sum())

table = work.pivot_table(index="Region", columns="Band", values="Sales",
                         aggfunc="sum", observed=True, fill_value=0)
print()
print(table.round(2).to_string())
print()
print("grand total: %.2f" % table.sum().sum())
print("raw total:   %.2f" % orders["Sales"].sum())

### Question 10

`get_dummies(orders, columns=["Segment"])` -> **raises** `KeyError: "None of [Index(['Segment'], dtype='str')] are in the [columns]"`. -> `Segment` is in `cust`, not `orders`.

The column exists — in the other file. That is the most common version of
this mistake and the message is unusually good about it, naming both what it
looked for and where it looked.

A fitting end to the day, because it is the same shape as everything else:
**Pandas is strict about structure and permissive about meaning.** A column
that is not there stops you at once. A `cut` whose top edge is too low, a
`map` that covers two categories out of four, a `qcut` on a column with five
distinct values, an outlier rule that discards 57% of revenue — none of
those raise anything at all.

The errors are not the hard part. The things that run are.

In [ ]:
print("orders columns:", list(orders.columns))
print("Segment is in customers:", "Segment" in cust.columns)
print(pd.get_dummies(orders, columns=["Segment"]))